# 6. Spectra + land cover (landcover)

Does the land cover improve the predictions? The same chain as notebook 3 (preprocessing `olif.snv.mc` refitted inside the model; c-org, n-tot, ph-water, clay; plsr, svr, randfor, cubist; test size 0.3) with `landcover` as a nominal covariate (one-hot encoded next to the spectra).

3 groups: cultivated, grassland, forest. The other classes (agroforest, lichens, spontan, shrubland, swamp) are left out (`"other": "exclude"`): their samples are not modelled and not predicted by the land cover models.

| Group | Classes |
|---|---|
| 1 | cultivated |
| 2 | grass-l-no-trees, grass-l-trees |
| 3 | conif-woodl, broadl-woodl, mixed-woodl |

A group is named by its classes joined with `|` in the tables of 6.3. Models are saved in `models/synthetic_calibration_10nm.olif.snv.mc_cv-landcover_tt/`. Run notebook 3 first (the spectra-only models are compared here).

In [ ]:
# Standard library imports
from os import path, getcwd

import sys

notebook_path = getcwd()

# Root of the xspatula_lucas repository (two levels up)
module_path = path.abspath(path.join('../..'))

if module_path not in sys.path:

    sys.path.append(module_path)

from src.lib import Initiate_process

from src.ai4sh import Run_process

## Scheme file

In [ ]:
scheme_file = '../scheme_synthetic.json'

## 6.1 Train and test with landcover

In [ ]:
process_file = 'project_synthetic/process/covariate_landcover/regression_modeling_landcover.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:

    Run_process(structured_process_D, scheme_params_D)

## 6.2 Independent validation: spectra only and spectra + landcover

Both model sets on the validation spectra; per indicator one row of spectra-only models and one row with landcover (prediction columns `<indicator>__<regressor>_tt` and `<indicator>__<regressor>+landcover_tt`).

In [ ]:
process_file = 'project_synthetic/process/covariate_landcover/predict_validation_landcover.json'

structured_process_D, scheme_params_D = Initiate_process(notebook_path, scheme_file, process_file)

if structured_process_D is not None:

    Run_process(structured_process_D, scheme_params_D)

## 6.3 Compare

r2 and RMSE of both model sets on the same validation samples (those the covariate models predict), `change` = with landcover − spectra only (negative RMSE change = better); then the RMSE per group: a covariate can help in a group where the spectra-only model is biased even when the overall metrics hardly change.

In [ ]:
# Spectra only vs spectra + covariate, on the samples both model sets predict
import json
import numpy as np
import pandas as pd

from src.ai4sh.machine_learning_model import _compute_metrics

covariate = 'landcover'
pred_dir  = 'data/predictions/synthetic_validation_10nm_landcover'
params    = json.load(open('process/covariate_landcover/regression_modeling_landcover.json'))['process'][0]['parameters']
grouping  = (params.get('class_grouping') or [{}])[0]
group_of  = {c: '|'.join(g) for g in grouping.get('groups', []) for c in g}

pred = pd.read_parquet(path.join(pred_dir, 'synthetic_validation_10nm_pred.parquet'))
# Group of each sample; None: no class or a class left out by class_grouping
pred['group'] = pred[covariate].map(lambda c: None if pd.isna(c) else group_of.get(str(c), None if group_of else str(c)))
models = json.load(open(path.join(pred_dir, 'synthetic_validation_10nm_pred.json')))['models']

overall, per_group = [], []
for m in models:
    indicator, rest = m['column'].split('__')
    if '+' in rest:
        continue
    regressor, mode = rest.rsplit('_', 1)
    cols = {'spectra': m['column'], '+' + covariate: '%s__%s+%s_%s' % (indicator, regressor, covariate, mode)}
    if cols['+' + covariate] not in pred.columns:
        continue
    ok = pred[indicator].notna() & pred[list(cols.values())].notna().all(axis=1)
    for model, col in cols.items():
        met = _compute_metrics(pred.loc[ok, indicator].values, pred.loc[ok, col].values)
        overall.append(dict(indicator=indicator, regressor=regressor, model=model, n=int(ok.sum()),
                            r2=met['r2'], rmse=met['rmse']))
        for g, sub in pred[ok].groupby('group'):
            per_group.append(dict(indicator=indicator, regressor=regressor, model=model, group=g, n=len(sub),
                                  rmse=float(np.sqrt(((sub[col] - sub[indicator]) ** 2).mean()))))

overall = pd.DataFrame(overall)
table = overall.pivot_table(index=['indicator', 'regressor'], columns='model', values=['r2', 'rmse'])
for v in ('r2', 'rmse'):
    table[(v, 'change')] = table[(v, '+' + covariate)] - table[(v, 'spectra')]
print('validation samples compared:', overall.groupby('indicator')['n'].max().to_dict())
display(table.round(3))

per_group = pd.DataFrame(per_group)
print('validation samples per group:', per_group.groupby('group')['n'].max().to_dict())
with pd.option_context('display.max_columns', None, 'display.width', 200):
    display(per_group.pivot_table(index=['indicator', 'regressor', 'model'], columns='group', values='rmse').round(3))
